# Segment Audio and Manifest with GCS Integration

This notebook reads a JSON manifest from GCS containing small audio snippets with ground truth labels, merges them into larger segments based on time proximity, and outputs:
1. A new JSON manifest in GCS with merged segments.
2. Segmented audio files uploaded to GCS.

This notebook reuses the GCS upload/download and file structure logic from `chirp_and_gemini_segment_audio.ipynb` but adds the snippet merging capability.

In [ ]:
# @title Imports and environment configuration
from collections import defaultdict
import json
import os
from pathlib import Path
import sys
from urllib.parse import urlparse

import ffmpeg
from google.cloud import storage
from google.colab import auth
from loguru import logger

# @markdown ### User Configuration
# @markdown Please enter your GCP project and bucket details below:
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = "one_hour_pilot"  # @param {type:"string"}
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}

# Merging configuration
MAX_GAP = 1.0  # @param {type:"number"}
MAX_DURATION = 30.0  # @param {type:"number"}

assert GCP_PROJECT_ID, "Please enter your GCP project ID"
assert GCS_BUCKET, "Please enter your GCS bucket name"
assert PROJECT_NAME, "PROJECT_NAME must be provided."

SOURCE_MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/manifests/{PROJECT_NAME}_transcriptions.json"
)
GCS_OUTPUT_PREFIX = f"segmented_audio/{PROJECT_NAME}_audio_merged"

LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segments"
BATCH_MANIFEST_FILENAME = "batch_manifest_merged.jsonl"

# Audio specs (used only if AUDIO_PREPROCESSING is True)
SAMPLE_RATE = 16000
CHANNELS = 1

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}")

In [ ]:
# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Helper functions and Merge Logic

def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Ensures the audio file from GCS is available locally in CACHE_DIR."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")

    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        bucket = gcs_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(str(local_path))
    return str(local_path)

def cleanup_gcs_output(output_prefix: str) -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    if blobs:
        logger.info(f"Cleaning existing files from gs://{GCS_BUCKET}/{output_prefix}...")
        bucket.delete_blobs(blobs)

def merge_snippets(snippets, max_gap=1.0, max_duration=30.0):
    """
    Merges small audio snippets into larger segments.
    
    Args:
        snippets: List of dicts containing audio_filepath, text, offset, duration.
        max_gap: Maximum silence gap (in seconds) allowed between snippets to merge them.
        max_duration: Maximum duration (in seconds) of a merged segment.
    """
    merged_data = []
    by_file = defaultdict(list)
    for s in snippets:
        by_file[s["audio_filepath"]].append(s)
        
    for filepath, file_snippets in by_file.items():
        file_snippets.sort(key=lambda x: x["offset"])
        current_segment = None
        
        for s in file_snippets:
            if current_segment is None:
                current_segment = s.copy()
                continue
                
            current_end = current_segment["offset"] + current_segment["duration"]
            gap = s["offset"] - current_end
            total_duration = s["offset"] + s["duration"] - current_segment["offset"]
            
            if gap <= max_gap and total_duration <= max_duration:
                current_segment["text"] += " " + s["text"]
                current_segment["duration"] = total_duration
            else:
                merged_data.append(current_segment)
                current_segment = s.copy()
                
        if current_segment:
            merged_data.append(current_segment)
            
    return merged_data

In [ ]:
# @title Create the merged segments and manifest file

def run_segmentation_pipeline() -> None:
    """Orchestrates extraction from the JSONL manifest in GCS after merging snippets."""
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

    current_output_prefix = GCS_OUTPUT_PREFIX if AUDIO_PREPROCESSING else f"{GCS_OUTPUT_PREFIX}_raw"
    cleanup_gcs_output(current_output_prefix)

    # 1. Load the JSONL manifest from GCS
    parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
    m_bucket = gcs_client.bucket(parsed_manifest.netloc)
    m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
    content = m_blob.download_as_text()

    manifest_data = [json.loads(line) for line in content.strip().split("\n")]

    # Merge snippets
    logger.info(f"Original snippets: {len(manifest_data)}")
    merged_manifest_data = merge_snippets(manifest_data, max_gap=MAX_GAP, max_duration=MAX_DURATION)
    logger.info(f"Merged segments: {len(merged_manifest_data)}")

    # Group by audio file
    files_to_process = {}
    for entry in merged_manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    output_bucket = gcs_client.bucket(GCS_BUCKET)
    final_manifest_entries = []

    # 2. Process each audio file and extract segments
    for gcs_audio_path, segments in files_to_process.items():
        local_src_path = ensure_local_gcs_audio(gcs_audio_path)
        example_id = Path(gcs_audio_path).stem

        logger.info(f"Processing {len(segments)} segments for: {example_id}")

        for i, seg in enumerate(segments):
            start_s = seg["offset"]
            duration_s = seg["duration"]
            seg_id = f"{i:03d}"
            filename = f"{example_id}__seg{seg_id}.flac"
            local_slice_path = Path(SEGMENTS_DIR) / filename

            # FFmpeg Command Construction for precise slicing
            stream = ffmpeg.input(local_src_path, ss=start_s, t=duration_s)

            if AUDIO_PREPROCESSING:
                stream = ffmpeg.output(stream, str(local_slice_path), ar=SAMPLE_RATE, ac=CHANNELS)
            else:
                stream = ffmpeg.output(stream, str(local_slice_path))

            ffmpeg.run(stream, overwrite_output=True, quiet=True)

            # Upload processed segment to GCS
            blob_name = f"{current_output_prefix}/{example_id}/{filename}"
            output_bucket.blob(blob_name).upload_from_filename(str(local_slice_path))

            # Collect metadata for the batch manifest
            final_manifest_entries.append({
                "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                "example_id": example_id,
                "offset": start_s,
                "duration": duration_s,
                "segment_id": seg_id,
                "text": seg.get("text", ""),
            })

    # 3. Create and upload the final batch manifest
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    gcs_manifest_path = f"{current_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    output_bucket.blob(gcs_manifest_path).upload_from_filename(str(local_manifest))
    logger.info(f"Pipeline Complete. {len(final_manifest_entries)} segments uploaded to {current_output_prefix}.")

run_segmentation_pipeline()